# MB1-E1 — Interval Re-evaluation of M0 vs M1

Evaluate `SOURCE_ANCHOR_FRAME` against frozen `LOCAL_RAW_CLIP_COARSE_TO_FINE` on the exact MB1 candidate windows. The AI intervals remain pseudo-GT (`human_reviewed=false`); preferred frames are secondary diagnostics only.

## INPUT CẦN GẮN TRÊN KAGGLE

1. Raw AIC videos: `/kaggle/input/datasets/nadkli/dataset-aic`
2. MB1 candidate pack containing `mb1_candidate_manifest.jsonl`
3. MB1 AI annotation pack containing `mb1_ai_semantic_moments.jsonl`
4. Stage 1B report containing `encoder/selected_encoder_contract.json`
5. Offline OpenAI CLIP ViT-B/32 asset containing `checkpoint/ViT-B-32.pt` and `source/openai_clip/`

Nested dataset roots are discovered automatically. Each root can also be overridden with the environment variables printed in the configuration cell.

## INPUT KHÔNG CẦN

Stage 1A vectors/index, Stage 1C/1D/1E, OPUS translator, RT1/RT2 benchmark, OCR, ASR, Objects, VLM, Event Graph, and Agent assets. No network/model download is performed by the experiment runner.

## OUTPUT ZIP

`/kaggle/working/triage_eg_mb1_e1_bundle.zip`


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = os.environ.get("AIC_REPO_URL", "https://github.com/Irthn1311/AIC2026_TeamPTK_SGU.git")
REPO_REF = os.environ.get("AIC_REPO_REF", "TRIAGEEG")
REPO_DIR = Path("/kaggle/working/AIC2026_TeamPTK_SGU")
if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True
    )
sys.path.insert(0, str(REPO_DIR / "src"))
COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True, text=True, check=True
).stdout.strip()
print("resolved commit:", COMMIT)

In [ ]:
DATASET_INPUT = Path(os.environ.get("AIC_DATA_ROOT", "/kaggle/input/datasets/nadkli/dataset-aic"))
CANDIDATE_INPUT = Path(
    os.environ.get(
        "AIC_MB1_CANDIDATE_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-mb1-candidates"
    )
)
ANNOTATION_INPUT = Path(
    os.environ.get(
        "AIC_MB1_ANNOTATION_ROOT", "/kaggle/input/datasets/irthn1311/triage-eg-mb1-ai-annotations"
    )
)
STAGE1B_INPUT = Path(
    os.environ.get(
        "AIC_STAGE1B_ROOT",
        "/kaggle/input/datasets/irthn1311/triage-eg-stage1b-encoder-compatibility-reports",
    )
)
CLIP_INPUT = Path(
    os.environ.get(
        "AIC_OPENAI_CLIP_ASSET_ROOT", "/kaggle/input/datasets/irthn1311/aic2026-openai-clip-vit-b32"
    )
)
OUTPUT_ROOT = Path("/kaggle/working/triage_eg_mb1_e1")
ZIP_PATH = Path("/kaggle/working/triage_eg_mb1_e1_bundle.zip")
print(
    {
        "raw": str(DATASET_INPUT),
        "candidates": str(CANDIDATE_INPUT),
        "annotations": str(ANNOTATION_INPUT),
        "stage1b": str(STAGE1B_INPUT),
        "clip": str(CLIP_INPUT),
        "output": str(OUTPUT_ROOT),
        "zip": str(ZIP_PATH),
    }
)

In [ ]:
SEARCH_ROOT = Path("/kaggle/input")


def resolve_unique_file(requested: Path, relative_marker: str) -> Path:
    requested = Path(requested)
    direct = requested / relative_marker
    if direct.is_file():
        return direct.resolve()
    if requested.is_file() and requested.name == Path(relative_marker).name:
        return requested.resolve()
    bases = [requested] if requested.is_dir() else []
    bases.append(SEARCH_ROOT)
    matches = []
    for base in bases:
        matches.extend(
            path.resolve()
            for path in base.rglob(Path(relative_marker).name)
            if path.is_file() and path.as_posix().endswith(relative_marker)
        )
    matches = sorted(set(matches))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one {relative_marker}; found {matches}")
    return matches[0]


def resolve_root_by_marker(requested: Path, relative_marker: str) -> Path:
    marker = resolve_unique_file(requested, relative_marker)
    root = marker
    for _ in Path(relative_marker).parts:
        root = root.parent
    return root.resolve()


def resolve_dataset_root(requested: Path, probe_video: str = "L23_V005.mp4") -> Path:
    direct = Path(requested) / "Videos_L23" / "video" / probe_video
    if direct.is_file():
        return Path(requested).resolve()
    roots = []
    for base in (Path(requested), SEARCH_ROOT):
        if base.is_dir():
            roots.extend(
                path.parents[2].resolve()
                for path in base.rglob(probe_video)
                if path.parent.name == "video"
            )
    roots = sorted(set(roots))
    if len(roots) != 1:
        raise RuntimeError(f"Expected one raw dataset root; found {roots}")
    return roots[0]


DATASET_ROOT = resolve_dataset_root(DATASET_INPUT)
CANDIDATE_PATH = resolve_unique_file(CANDIDATE_INPUT, "mb1_candidate_manifest.jsonl")
ANNOTATION_PATH = resolve_unique_file(ANNOTATION_INPUT, "mb1_ai_semantic_moments.jsonl")
STAGE1B_ROOT = resolve_root_by_marker(STAGE1B_INPUT, "encoder/selected_encoder_contract.json")
CLIP_ROOT = resolve_root_by_marker(CLIP_INPUT, "checkpoint/ViT-B-32.pt")
print(
    json.dumps(
        {
            "dataset_root": str(DATASET_ROOT),
            "candidate_manifest": str(CANDIDATE_PATH),
            "annotations": str(ANNOTATION_PATH),
            "stage1b_root": str(STAGE1B_ROOT),
            "clip_root": str(CLIP_ROOT),
        },
        indent=2,
    )
)

In [ ]:
from triage_eg.experiments.mb1_e1 import MB1E1Config, preflight_mb1_e1

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
CONFIG = MB1E1Config(
    dataset_root=DATASET_ROOT,
    candidate_manifest_path=CANDIDATE_PATH,
    annotation_path=ANNOTATION_PATH,
    stage1b_root=STAGE1B_ROOT,
    clip_asset_root=CLIP_ROOT,
    output_root=OUTPUT_ROOT,
    seed=2026,
    device=os.environ.get("AIC_CLIP_DEVICE", "auto"),
    batch_size=int(os.environ.get("AIC_CLIP_BATCH_SIZE", "16")),
    build_git_commit=COMMIT,
)
PREFLIGHT = preflight_mb1_e1(CONFIG)
print(json.dumps(PREFLIGHT, indent=2))

In [ ]:
from triage_eg.experiments.mb1_e1 import run_mb1_e1

RESULT = run_mb1_e1(CONFIG)
print(json.dumps(RESULT["summary"], indent=2))

In [ ]:
primary = RESULT["metrics"]["ALL_HIGH_MEDIUM"]
print(json.dumps(primary, indent=2))
print("M1_INTERVAL_QUALITY_DECISION = NOT_EVALUATED")
print("Reason: interval pseudo-GT is AI-generated and human_reviewed=false.")

In [ ]:
from triage_eg.experiments.mb1_e1 import create_mb1_e1_bundle

bundle = create_mb1_e1_bundle(OUTPUT_ROOT, ZIP_PATH)
print("MB1_E1_IMPLEMENTATION_STATUS = COMPLETE")
print("MB1_E1_REAL_STATUS = COMPLETE")
print("INTERVAL_BENCHMARK_STATUS = VALID")
print("DOWNLOAD ZIP:", bundle, "size_bytes=", bundle.stat().st_size)